In [14]:
function dCov_M_slow(x,y)
    n = length(x)
    A = zeros(Float64,n,n)
    B = zeros(Float64,n,n)
    for i in 1:n, j in 1:n
        A[i,j] = abs(x[i] - x[j])
        B[i,j] = abs(y[i] - y[j])
    end
    A = A .- mean(A,dims=1) .- mean(A,dims=2) .+ mean(A)
    B = B .- mean(B,dims=1) .- mean(B,dims=2) .+ mean(B)
    return mean(A .* B)
end

function dCor_M_slow(x,y)
    return dCov_M_slow(x,y) / sqrt(dCov_M_slow(x,x) * dCov_M_slow(y,y))
end

function dCor_M_fast(x,y)
    n = length(x)
    A = zeros(Float64,n,n)
    B = zeros(Float64,n,n)
    for i in 1:n, j in 1:n
        A[i,j] = abs(x[i] - x[j])
        B[i,j] = abs(y[i] - y[j])
    end
    A .= A .- mean(A,dims=1) .- mean(A,dims=2) .+ mean(A)
    B .= B .- mean(B,dims=1) .- mean(B,dims=2) .+ mean(B)
return mean(A .* B) / sqrt(mean(A .* A) * mean(B .* B))
end

function dCor_M_half(x,y)
    n = length(x)
    A = zeros(Int8, n, n)
    B = zeros(Int8, n, n)
    wektor_A = zeros(Float64, n)
    wektor_B = zeros(Float64, n)

    @inbounds for j in 2:n, i in 1:j-1
        a = abs(x[i] - x[j])
        b = abs(y[i] - y[j])
        A[i,j] = a
        B[i,j] = b
        wektor_A[i] += a
        wektor_A[j] += a
        wektor_B[i] += b
        wektor_B[j] += b
    end

    średnia_A = sum(wektor_A) / n^2
    średnia_B = sum(wektor_B) / n^2
    wektor_A ./= n
    wektor_B ./= n
    dcov_XY = 0.0
    dcov_XX = 0.0
    dcov_YY = 0.0

    @inbounds for j in 2:n, i in 1:j-1
        a = A[i,j] - wektor_A[i] - wektor_A[j] + średnia_A
        b = B[i,j] - wektor_B[i] - wektor_B[j] + średnia_B 
        dcov_XY += 2*a*b
        dcov_XX += 2*a^2
        dcov_YY += 2*b^2
    end
    @inbounds for k in 1:n
        a = średnia_A - 2*wektor_A[k]
        b = średnia_B - 2*wektor_B[k]
        dcov_XY += a*b
        dcov_XX += a^2
        dcov_YY += b^2
    end

    return dcov_XY / sqrt(dcov_XX * dcov_YY)
end

function dCor_M_final(x,y)
    n = length(x)
    wektor_A = zeros(Float64, n)
    wektor_B = zeros(Float64, n)

    @inbounds for j in 2:n, i in 1:j-1
        a = abs(x[i] - x[j])
        b = abs(y[i] - y[j])
        wektor_A[i] += a
        wektor_A[j] += a
        wektor_B[i] += b
        wektor_B[j] += b
    end

    średnia_A = sum(wektor_A) / n^2
    średnia_B = sum(wektor_B) / n^2
    wektor_A ./= n
    wektor_B ./= n
    dcov_XY = 0.0
    dcov_XX = 0.0
    dcov_YY = 0.0

    @inbounds for j in 2:n, i in 1:j-1
        distA = abs(x[i] - x[j])
        distB = abs(y[i] - y[j]) 
        a = distA - wektor_A[i] - wektor_A[j] + średnia_A 
        b = distB - wektor_B[i] - wektor_B[j] + średnia_B
        dcov_XY += 2*a*b
        dcov_XX += 2*a^2
        dcov_YY += 2*b^2
    end
    @inbounds for k in 1:n
        a = średnia_A - 2*wektor_A[k]
        b = średnia_B - 2*wektor_B[k]
        dcov_XY += a*b
        dcov_XX += a^2
        dcov_YY += b^2
    end

    return dcov_XY / sqrt(dcov_XX * dcov_YY)
end

function bin_U(x)
    Bins = Vector{Int8}(undef, length(x)) 
    @inbounds @simd for i in eachindex(x) 
        Bins[i] = trunc(Int8, 5*x[i]) + 1 
    end 
    return Bins
end

function bin_N(x)
    Bins = Vector{Int8}(undef, length(x)) 

    @inbounds for i in eachindex(x)
        xi = x[i]

        if xi < 0
            if xi < -1
                if xi < -2
                    Bins[i] = 1
                else
                    Bins[i] = 2
                end
            else
                Bins[i] = 3
            end
        else
            if xi < 1
                Bins[i] = 4
            elseif xi < 2
                Bins[i] = 5
            else
                Bins[i] = 6
            end
        end
    end

    return Bins
end

function tabela_K(bu,bn)
    T = zeros(Int, maximum(bu), maximum(bn))

    @inbounds for i in eachindex(bu, bn)
        T[bu[i], bn[i]] += 1
    end

    return T
end

function tabela_K!(T,bu,bn)
    T .= 0
    @inbounds for i in eachindex(bu, bn)
        T[bu[i], bn[i]] += 1
    end
    return T
end

tabela_K! (generic function with 1 method)

In [15]:
#Testy prędkości 
using BenchmarkTools
using Random
Random.seed!(1234)
x = rand(100)
y = randn(100)

X = bin_U(x)
Y = bin_N(y)

@benchmark tabela_K(X,Y)

BenchmarkTools.Trial: 10000 samples with 839 evaluations per sample.
 Range (min … max):  142.074 ns …   3.242 μs  ┊ GC (min … max): 0.00% … 92.32%
 Time  (median):     155.066 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   191.829 ns ± 149.501 ns  ┊ GC (mean ± σ):  9.38% ± 11.14%

  █▆▅▅▃▂▂▁▁▂▂▂                                                  ▁
  █████████████▇▇▆▅▆▅▆▆▆▄▄▃▃▅▄▁▁▁▃▃▁▁▁▁▁▁▁▃▁▁▁▃▁▁▄▄▆▅▄▅▄▅▅▆▅▆▆▆ █
  142 ns        Histogram: log(frequency) by time       1.03 μs <

 Memory estimate: 320 bytes, allocs estimate: 2.

In [16]:
T = zeros(Int, maximum(X), maximum(Y))
@benchmark tabela_K!(T,X,Y)

BenchmarkTools.Trial: 10000 samples with 943 evaluations per sample.
 Range (min … max):   97.879 ns …  33.215 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     105.938 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   119.771 ns ± 340.024 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▅▇██▆▄▂ ▂▄▄▃▂▂▁▃▃▂▃▃▁                                         ▂
  ████████████████████████▇▇███▆▅▇▆▆▆▆▇▆▅▅▆▄▆▅▅▇▆▄▅▅▄▄▅▄▆▆▇▅▅▅▆ █
  97.9 ns       Histogram: log(frequency) by time        236 ns <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [17]:
@benchmark dCor_M_slow(X,Y)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  169.200 μs … 138.645 ms  ┊ GC (min … max):  0.00% … 99.83%
 Time  (median):     211.900 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   431.007 μs ±   3.102 ms  ┊ GC (mean ± σ):  34.11% ± 17.46%

  ██▆▅▄▄▃▃▂▂▁      ▂▃▃▂▁▁▁ ▁      ▁▁▁                           ▂
  ████████████▇▇▇▆█████████████▇▇▇█████▇▇▇▆▆▆▆▇▆▆▆▅▆▅▄▅▆▄▄▂▃▄▄▄ █
  169 μs        Histogram: log(frequency) by time       1.54 ms <

 Memory estimate: 1.16 MiB, allocs estimate: 142.

In [18]:
@benchmark dCor_M_fast(X,Y)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):   73.300 μs … 126.373 ms  ┊ GC (min … max):  0.00% … 99.88%
 Time  (median):      86.400 μs               ┊ GC (median):     0.00%
 Time  (mean ± σ):   175.575 μs ±   2.148 ms  ┊ GC (mean ± σ):  35.81% ± 11.65%

  █▆▅▄▃▂▁ ▁▃▄▃▂▁▁                                               ▂
  ███████████████████▇▇▆▅▅▄▄▄▃▄▁▁▄▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▆▆▇▅▇▆▅▄▅ █
  73.3 μs       Histogram: log(frequency) by time        888 μs <

 Memory estimate: 396.36 KiB, allocs estimate: 48.

In [19]:
@benchmark dCor_M_half(X,Y)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  25.600 μs … 907.300 μs  ┊ GC (min … max): 0.00% … 91.23%
 Time  (median):     26.900 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   30.975 μs ±  33.137 μs  ┊ GC (mean ± σ):  4.57% ±  4.24%

  ▇█▇▅▄▃▁▁ ▁                ▁  ▁▁▁                             ▂
  ████████████▇▇▆▆▇▇▆▆▆▇███████████▇█▇▇▄▆▅▆▃▆▄▅▆▅▆▄▅▆▄▆▇▇▅▄▄▄▅ █
  25.6 μs       Histogram: log(frequency) by time      64.7 μs <

 Memory estimate: 21.53 KiB, allocs estimate: 9.

In [20]:
@benchmark dCor_M_final(X,Y)

BenchmarkTools.Trial: 10000 samples with 1 evaluation per sample.
 Range (min … max):  23.100 μs … 999.300 μs  ┊ GC (min … max): 0.00% … 94.23%
 Time  (median):     24.100 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   26.856 μs ±  15.125 μs  ┊ GC (mean ± σ):  0.72% ±  1.59%

  ▆█▆▄▁▁                    ▁▁▁▂▁▂ ▁ ▁ ▁                       ▁
  ████████▇▇▇▆▆▆▆▅▅▆▅▆▆▇▇▇▇███████████▇█▇▅▄▄▄▄▃▃▂▂▂▄▄▄▂▂▂▄▄▂▃▄ █
  23.1 μs       Histogram: log(frequency) by time      54.8 μs <

 Memory estimate: 1.83 KiB, allocs estimate: 5.